# Tendencia de clustering (Hopkins y VAT) · Perfiles de votantes

**Proyecto:** Hackathon INE 2026 · Reto 1 · Perfiles ciudadanos de participación.

## Pregunta que responde
Antes de agrupar (K-means, Ward, mezclas gaussianas), ¿los datos **tienen** estructura de grupos, o solo forman un continuo? Esta libreta responde con dos pruebas:

- **Hopkins**: un número entre 0 y 1 que compara qué tan cerca están los puntos reales entre sí contra puntos aleatorios.
- **VAT** (*Visual Assessment of cluster Tendency*): una imagen de la matriz de distancias reordenada. Los grupos aparecen como bloques oscuros en la diagonal.

## Dos análisis
- **A. Por región**: un test para cada región (norte, centro, sur).
- **B. Control conjunto**: el mismo test con las tres regiones juntas. La región **no** entra como variable; sirve para ver si los grupos se mezclan entre zonas o se separan por zona.

## Alcance
Entra: construir la tabla de análisis, medir la tendencia y revisar sesgos de lectura. **No entra**: el clustering en sí, la elección de k ni la validación de perfiles (siguiente paso, reutilizando `estimator()` de `scripts/perfiles_sociodemograficos.py`).

## Índice
1. Glosario · 2. Decisiones y supuestos · 3. Cómo leer Hopkins y VAT · 4. Preparación del entorno · 5. Carga y validación · 6. Tabla de análisis · 7. Filtros de calidad · 8. Funciones Hopkins y VAT · 9. Controles y variantes · 10. Resultados (A y B) · 11. Imágenes VAT · 12. ¿Los grupos siguen a los estados? · 13. Autopruebas · 14. Conclusiones

## Reproducir el entorno (Anaconda)
```
conda create -n civia python=3.12 pandas numpy scipy scikit-learn matplotlib jupyterlab
conda activate civia
jupyter lab
```
Entrada: `data/processed/civia_dataset_maestro_municipal_2009_2024_con_region.csv` (solo local). Salidas: `reports/local/tendencia/` (ignorada por Git).

## 1. Glosario
| Término | Significado aquí |
| --- | --- |
| **Unidad (fila)** | Una combinación **entidad × sexo × grupo de edad × tipo de sección**, por ejemplo "Jalisco, mujer, 25-34, urbana". Es un grupo de personas, no una persona. |
| **Feature (columna)** | Un número que describe a la fila. Aquí son 6: la tasa de participación en cada elección (2009, 2012, 2015, 2018, 2021, 2024). |
| **Tasa de participación** | `si_votaron / (si_votaron + no_votaron)`. Se recalcula desde los conteos; nunca se promedian porcentajes. |
| **NS (no especificados)** | Registros sin dato de participación (cuadernillo sin marcas o no disponible). **No es abstención.** Se usa solo para filtrar filas poco confiables, no como feature. |
| **Escalado** | Restar la media y dividir entre la desviación de cada columna, para que ninguna elección pese más por su nivel. |
| **Hopkins (H)** | Cerca de 0.5: datos como aleatorios. Cerca de 1: los puntos se agrupan. |
| **VAT** | Imagen de distancias entre filas reordenadas por cercanía. Bloques oscuros cuadrados en la diagonal = grupos. |
| **Gradiente** | Los datos varían de forma continua (p. ej. la participación sube con la edad) sin cortes naturales. No es un grupo. |
| **Efecto entidad** | Parte de la variación que se explica solo por el estado. Si domina, el clustering encontraría "estados" y no perfiles de votante. |
| **Línea base (nula)** | Datos sin grupos, con las mismas columnas, contra los que se compara el resultado real. |

## 2. Decisiones y supuestos
- **Unidad = entidad × sexo × edad × sección.** Con solo sexo × edad × sección habría 48 filas por región, muy pocas para Hopkins. Con la entidad hay unas 380 a 620 por región.
- **Features = 6 tasas de participación.** Con una sola elección quedarían 1 o 2 columnas útiles. La trayectoria de 6 elecciones da información suficiente. Sexo, edad y sección **no** son features: definen quién es la fila y se usan después para describir.
- **Se usa `entidad_clave`**, no el nombre: "Distrito Federal" (antes de 2015) y "Ciudad de México" son la misma entidad (clave 9).
- **No binario queda fuera**: solo existe en 2024 y no tiene trayectoria de 6 elecciones. Se reporta aparte en `reports/sociodemografico/no_binario_descriptivo.csv`.
- **Filtro de calidad** (mismo criterio de `scripts/perfiles_sociodemograficos.py`): tasa disponible en las 6 elecciones, al menos 100 registros especificados (`si + no`) y NS ≤ 20 % en cada una.
- **Región** es la propuesta propia del proyecto (norte, centro, sur), asignada por entidad completa; no es una división oficial del INEGI.
- **Lista nominal** no es feature: haría que los grupos salgan por tamaño de población y no por comportamiento.
- **Semilla fija (42)** en todos los pasos aleatorios, para que los resultados se repitan.
- **Limitación:** son cortes repetidos de grupos, no seguimiento de las mismas personas. No se atribuyen causas.

## 3. Cómo leer Hopkins y VAT
| Hopkins (H) | Lectura |
| --- | --- |
| ≈ 0.5 | Sin estructura (aleatorio). |
| 0.6 – 0.75 | Tendencia moderada. |
| > 0.75 | Tendencia clara. |

**Hopkins solo cuenta si supera la línea base.** Las 6 tasas están muy correlacionadas (casi todo es "participa más o menos"). Eso hace que los puntos formen una nube alargada y Hopkins sube **aunque no haya grupos**. Por eso cada resultado se compara con dos líneas base:
- **Gaussiana con la misma media y covarianza**: una nube sin grupos con la misma forma alargada. Es la comparación principal.
- **Columnas permutadas**: rompe también la correlación. Es una referencia más laxa.

**Veredicto automático** (regla heurística, ajustable en la celda de resultados): con `delta = H_real − H_gaussiana`:
- **clara** si H > 0.75 y delta > 0.10;
- **moderada** si H > 0.60 y delta > 0.05;
- **sin evidencia** en cualquier otro caso (gradiente o aleatorio).

**VAT:** se muestran tres paneles por caso: datos reales, la nube gaussiana nula y los datos sin efecto entidad. Bloques cuadrados oscuros en la diagonal sugieren grupos; un degradado continuo sugiere gradiente. Si el panel real se parece al nulo, no hay grupos.

**Variantes que se prueban:** `original` (6 columnas escaladas), `PCA 3 comp.` (reduce la redundancia entre elecciones) y `sin efecto entidad` (a cada tasa se le resta la media de su entidad, para ver si queda estructura dentro de los estados).

## 4. Preparación del entorno
Importaciones, semilla global, rutas y versiones de las librerías (para reproducir).

In [ ]:
import platform
from pathlib import Path
import numpy as np, pandas as pd, scipy, sklearn, matplotlib
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
from scipy.spatial.distance import pdist, squareform
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import normalized_mutual_info_score as nmi

SEED = 42
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data/processed').exists())
DATA = ROOT / 'data/processed/civia_dataset_maestro_municipal_2009_2024_con_region.csv'
OUT = ROOT / 'reports/local/tendencia'
OUT.mkdir(parents=True, exist_ok=True)

YEARS = [2009, 2012, 2015, 2018, 2021, 2024]
FEATURES = [f'part_{y}' for y in YEARS]
REGIONS = ['norte', 'centro', 'sur']
COUNTS = ['lista_nominal', 'si_votaron', 'no_votaron', 'no_especificados']
MIN_ESP, MAX_NS = 100, 20     # filtro de calidad (ver sección 7)
REPS = 50                     # repeticiones de Hopkins

print(f'Python {platform.python_version()} · numpy {np.__version__} · pandas {pd.__version__} · '
      f'scipy {scipy.__version__} · scikit-learn {sklearn.__version__} · matplotlib {matplotlib.__version__}')
print('Raíz del proyecto:', ROOT)

## 5. Carga y validación
Se lee el maestro municipal y se repiten las comprobaciones del script del repo: `LN = SV + NV + NS` en todas las filas, sin duplicados, y cada entidad pertenece a una sola región. Después se excluye **No binario**.

In [ ]:
d = pd.read_csv(DATA, encoding='utf-8-sig')
assert (d.lista_nominal == d.si_votaron + d.no_votaron + d.no_especificados).all()
assert not d.duplicated(['anio', 'entidad_clave', 'municipio_clave', 'sexo_codigo', 'grupo_edad', 'tipo_seccion_codigo']).any()
assert d.groupby('entidad_clave').region.nunique().max() == 1
print(f'{len(d):,} filas municipales · {d.entidad_clave.nunique()} entidades · años {sorted(d.anio.unique())}')
print(f'Se excluyen {(d.sexo == "No binario").sum()} filas de No binario (solo existen en 2024).')
d = d[d.sexo != 'No binario']
d.groupby('region').entidad.agg(lambda s: sorted(s.unique())).to_frame('entidades (nombres)')

## 6. Tabla de análisis
Los conteos municipales se **suman** por entidad × sexo × grupo de edad × tipo de sección × año. La tasa se **recalcula** desde las sumas. Luego se pasa a formato ancho: una fila por unidad y una columna por elección (`part_2009 … part_2024`).

In [ ]:
KEY = ['region', 'entidad_clave', 'sexo', 'grupo_edad', 'tipo_seccion_codigo']
a = d.groupby([*KEY, 'anio'], observed=True)[COUNTS].sum()
a['esp'] = a.si_votaron + a.no_votaron
a['part'] = a.si_votaron / a.esp.where(a.esp > 0)
a['ns_pct'] = 100 * a.no_especificados / a.lista_nominal.where(a.lista_nominal > 0)
wide = {k: a[k].unstack('anio').reindex(columns=YEARS) for k in ['part', 'esp', 'ns_pct']}
T = wide['part'].copy()
T.columns = FEATURES
print(f'{len(T):,} unidades (antes de filtros)')
T.head()

## 7. Filtros de calidad
Se conservan las unidades con tasa en las 6 elecciones, **al menos 100 registros especificados** y **NS ≤ 20 %** en todas. Una tasa calculada con pocos registros es ruido, y Hopkins lo leería como estructura. NS solo filtra; no se agrupa por él.

In [ ]:
ok = T.notna().all(axis=1) & wide['esp'].ge(MIN_ESP).all(axis=1) & wide['ns_pct'].le(MAX_NS).all(axis=1)
perdidas = pd.DataFrame({'antes': T.groupby('region').size(), 'despues': T[ok].groupby('region').size()}).reindex(REGIONS)
perdidas['perdidas_%'] = (100 * (1 - perdidas.despues / perdidas.antes)).round(1)
T = T[ok]
cases = {r: T.xs(r, level='region', drop_level=False) for r in REGIONS}   # análisis A
cases['conjunto'] = T                                                      # análisis B (control)
perdidas

## 8. Funciones Hopkins y VAT
Se escriben aquí (≈30 líneas) porque los paquetes específicos están sin mantenimiento.

- **`hopkins`**: toma `m` puntos reales (≈10 % de n) y mide su distancia al vecino más cercano (`w`); genera `m` puntos uniformes dentro de la caja que contiene los datos y mide su distancia al punto real más cercano (`u`). `H = Σu / (Σu + Σw)`. Si los datos se agrupan, `w` es chico y H sube hacia 1. Se repite `REPS` veces y se reporta media y desviación.
- **`vat`**: calcula la matriz de distancias euclidianas y reordena filas y columnas con un recorrido tipo Prim (siempre se añade el punto más cercano al conjunto ya ordenado). Los grupos quedan como bloques contiguos.

In [ ]:
def hopkins(X, m=None, reps=REPS, seed=SEED):
    """Estadístico de Hopkins con intervalo empírico.

    X: matriz (n, d) ya escalada. m: tamaño de muestra (por defecto 10 % de n, mínimo 10).
    reps: repeticiones con muestras distintas. seed: semilla (resultados reproducibles).
    Devuelve (media, desviación estándar) de H en las repeticiones.
    """
    rng = np.random.default_rng(seed)
    n, dim = X.shape
    m = m or max(10, n // 10)
    tree = cKDTree(X)
    lo, hi = X.min(0), X.max(0)
    hs = []
    for _ in range(reps):
        w = tree.query(X[rng.choice(n, m, replace=False)], k=2)[0][:, 1]   # vecino más cercano distinto de sí mismo
        u = tree.query(rng.uniform(lo, hi, (m, dim)), k=1)[0]
        hs.append(u.sum() / (u.sum() + w.sum()))
    return float(np.mean(hs)), float(np.std(hs))


def vat(X):
    """Matriz de distancias reordenada (VAT).

    X: matriz (n, d). Devuelve (matriz reordenada n×n, orden de filas).
    """
    D = squareform(pdist(X))
    n = len(D)
    first = int(np.unravel_index(D.argmax(), D.shape)[0])
    order, rest = [first], np.ones(n, bool)
    rest[first] = False
    near = D[first].copy()
    for _ in range(n - 1):
        j = int(np.argmin(np.where(rest, near, np.inf)))
        order.append(j)
        rest[j] = False
        near = np.minimum(near, D[j])
    return D[np.ix_(order, order)], np.array(order)

## 9. Controles y variantes
Para cada caso (norte, centro, sur y conjunto) se calcula Hopkins en tres representaciones y contra tres referencias:

| | |
| --- | --- |
| **Representaciones** | `original`, `PCA 3 comp.`, `sin efecto entidad` |
| **Referencias** | datos reales · gaussiana con la misma media y covarianza · columnas permutadas |

También se mide el **efecto entidad** como η² por feature: la fracción de la varianza total que explica solo la entidad. Un η² alto (≈ 0.5 o más) avisa que el estado domina la variación.

In [ ]:
sc = lambda M_: StandardScaler().fit_transform(M_)


def variantes(frame):
    """Tres representaciones de la misma tabla: escalada, PCA a 3 componentes y sin la media de la entidad."""
    raw = frame[FEATURES]
    ent = frame.index.get_level_values('entidad_clave')
    X = sc(raw)
    return {'original': X,
            'PCA 3 comp.': PCA(3, random_state=SEED).fit_transform(X),
            'sin efecto entidad': sc(raw - raw.groupby(ent).transform('mean'))}


def nulos(X, seed=SEED):
    """Dos referencias sin grupos: gaussiana con igual media/covarianza y columnas permutadas."""
    rng = np.random.default_rng(seed)
    return {'gauss': rng.multivariate_normal(X.mean(0), np.cov(X.T), len(X)),
            'perm': np.column_stack([rng.permutation(c) for c in X.T])}


def eta2(frame):
    """Fracción de la varianza de cada feature que explica la entidad."""
    raw = frame[FEATURES]
    g = raw.groupby(frame.index.get_level_values('entidad_clave')).transform('mean')
    return ((g - raw.mean()) ** 2).sum() / ((raw - raw.mean()) ** 2).sum()


def veredicto(h, delta):
    """Regla heurística (ver sección 3); los umbrales son exploratorios."""
    if h > 0.75 and delta > 0.10: return 'clara'
    if h > 0.60 and delta > 0.05: return 'moderada'
    return "sin evidencia"

## 10. Resultados · análisis A (norte, centro, sur) y B (conjunto)
Cada fila es un caso × representación. `delta` es `H_real − H_gauss`: cuánta tendencia hay **más allá de lo que explica la forma alargada** de los datos.

In [ ]:
rows, eta_rows, X_cases = [], [], {}
for case, frame in cases.items():
    eta_rows.append({'caso': case, **eta2(frame).round(3).rename(lambda c: c.replace('part_', 'η² ')).to_dict()})
    for var, X in variantes(frame).items():
        if var == 'original': X_cases[case] = X
        h, sd = hopkins(X)
        nul = nulos(X)
        hg, hp = hopkins(nul['gauss'])[0], hopkins(nul['perm'])[0]
        rows.append({'caso': case, 'variante': var, 'n': len(X), 'H_real': round(h, 3), 'sd': round(sd, 3),
                     'H_gauss': round(hg, 3), 'H_perm': round(hp, 3), 'delta': round(h - hg, 3), 'veredicto': veredicto(h, h - hg)})
resumen = pd.DataFrame(rows)
resumen.to_csv(OUT / 'resumen_hopkins.csv', index=False)
eta = pd.DataFrame(eta_rows).set_index('caso')
eta.to_csv(OUT / 'efecto_entidad_eta2.csv')
print('Efecto entidad (fracción de varianza explicada por la entidad):')
display(eta)
print('Hopkins:')
resumen

## 11. Imágenes VAT
Tres paneles por caso: datos reales (`original`), nube gaussiana nula con la misma forma y datos sin efecto entidad. **Oscuro = cercano.** Bloques cuadrados oscuros en la diagonal indicarían grupos; si el panel real se parece al nulo, no hay grupos. Se guardan como PNG en `reports/local/tendencia/`.

In [ ]:
for case, frame in cases.items():
    X = X_cases[case]
    paneles = [(X, 'datos reales'), (nulos(X)['gauss'], 'nulo gaussiano (misma covarianza)'),
               (variantes(frame)['sin efecto entidad'], 'sin efecto entidad')]
    fig, axs = plt.subplots(1, 3, figsize=(15, 5))
    for ax, (Z, titulo) in zip(axs, paneles):
        ax.imshow(vat(Z)[0], cmap='gray', interpolation='nearest')
        ax.set(title=titulo, xticks=[], yticks=[])
    fig.suptitle(f'VAT · {case} · n={len(X)}', y=1.03)
    fig.tight_layout()
    fig.savefig(OUT / f'vat_{case}.png', dpi=150, bbox_inches='tight')
    plt.show()

## 12. ¿Los grupos siguen a los estados?
Diagnóstico, **no** es el clustering final: se corta cada caso en 2, 3 y 4 grupos con Ward y se mide qué tanto coinciden con cada identificador usando información mutua normalizada (NMI: 0 = nada que ver, 1 = idénticos).

Si la NMI con `entidad_clave` es mucho mayor que con sexo, edad o sección, el clustering estaría encontrando **estados** y no perfiles de votante. En ese caso hay que usar la variante `sin efecto entidad`.

In [ ]:
ids = {'entidad': 'entidad_clave', 'sexo': 'sexo', 'edad': 'grupo_edad', 'sección': 'tipo_seccion_codigo'}
diag = []
for case, frame in cases.items():
    for k in (2, 3, 4):
        lab = AgglomerativeClustering(n_clusters=k, linkage='ward').fit_predict(X_cases[case])
        diag.append({'caso': case, 'k': k, **{n: round(nmi(frame.index.get_level_values(c), lab), 3) for n, c in ids.items()}})
diag = pd.DataFrame(diag)
diag.to_csv(OUT / 'diagnostico_nmi.csv', index=False)
diag

## 13. Autopruebas
Comprobaciones mínimas de que las funciones y los datos están bien. Si alguna falla, no se debe confiar en los resultados de arriba.
1. Hopkins sobre datos uniformes aleatorios debe dar ≈ 0.5.
2. Hopkins sobre dos nubes bien separadas debe dar > 0.85.
3. `vat` devuelve una permutación válida de las filas.
4. Una celda de la tabla coincide con el CSV crudo (Aguascalientes 2009, hombres, 18-24, urbana, sumando todos sus municipios). Se omite si los crudos no están descomprimidos.

In [ ]:
rng = np.random.default_rng(SEED)
h_unif = hopkins(rng.uniform(size=(600, 3)))[0]
h_blobs = hopkins(np.vstack([rng.normal(0, .3, (300, 3)), rng.normal(5, .3, (300, 3))]))[0]
assert 0.4 < h_unif < 0.6, h_unif
assert h_blobs > 0.85, h_blobs
orden = vat(rng.normal(size=(50, 3)))[1]
assert sorted(orden) == list(range(50))
print(f'Hopkins uniforme={h_unif:.3f} (≈0.5) · dos nubes={h_blobs:.3f} (>0.85) · VAT ok')

crudo = ROOT / 'data/raw/DatosAbiertos_DECEyEC_ConteosCensales2009/datosabiertos_deceyec_conteoscensales2009_ags.csv'
if crudo.exists():
    r = pd.read_csv(crudo, encoding='utf-8-sig', low_memory=False)
    r = r[(r.SEXO == 0) & r.EDAD.between(18, 24) & (r.TIPOSEC == 'U')]
    mio = a.loc[('norte', 1, 'Hombre', '18-24', 'U', 2009), COUNTS]
    assert list(mio) == [r.LN.sum(), r.SV.sum(), r.NV.sum(), r.NS.sum()], (list(mio), r[['LN', 'SV', 'NV', 'NS']].sum().tolist())
    print('Celda Aguascalientes 2009 coincide con el CSV crudo:', mio.to_dict())
else:
    print('Crudos no encontrados; se omite la comparación con el CSV original.')

## 14. Conclusiones
_Resultados de la ejecución con el maestro local del 2026-10-28 (semilla 42, 50 repeticiones). Si cambian los datos o los filtros, volver a ejecutar y revisar este texto._

**Veredicto: no hay evidencia de grupos naturales en ninguna región ni en el control conjunto.** Los 12 casos (4 casos × 3 representaciones) salen `sin evidencia`.

1. **Hopkins es alto (0.86 a 0.89), pero no significa grupos.** La nube gaussiana sin grupos y con la misma covarianza da prácticamente lo mismo (`delta` entre −0.017 y +0.012; el máximo es +0.035 en el conjunto con PCA). El valor alto viene de que las 6 tasas están muy correlacionadas y los puntos forman una nube alargada.
2. **VAT sin bloques claros.** El panel real no muestra bloques cuadrados oscuros en la diagonal y se parece al nulo gaussiano. Los datos se leen como un continuo.
3. **El estado no domina.** η² por entidad va de 0.10 a 0.40 según la elección. Quitar el efecto entidad no cambia el veredicto.
4. **Lo que sí ordena los datos es la edad.** Al cortar con Ward en 2 a 4 grupos, la coincidencia (NMI) es de 0.19 a 0.29 con el grupo de edad, de 0.03 a 0.12 con la entidad y de 0.02 o menos con sexo y sección. La estructura es sobre todo un **gradiente por edad**.
5. **El control conjunto da lo mismo que cada región.** La región no separa perfiles de forma visible.

**Qué implica para el siguiente paso.** Se puede segmentar (K-means, Ward, mezclas gaussianas), pero hay que presentar los perfiles como **tipologías útiles para comparar y describir**, no como grupos que existan por sí mismos en los datos. La validación de k (estabilidad por submuestras, silhouette) pasa a ser el criterio principal, y los resultados se deben reportar con esa cautela.

**Para profundizar (no hecho aquí):**
- Probar otra representación de las filas, por ejemplo sin la edad como eje dominante (dentro de cada grupo de edad), para ver si sexo y sección forman grupos.
- Contrastar contra la agregación municipal (cientos de unidades por región).
- Los umbrales del veredicto son exploratorios; no son una prueba estadística formal.
